# BPE Tokenizer — Deep Dive

This notebook walks through **every function** of `code/bpe_tokenizer.py`, from raw bytes to trained vocab.  
Run cells top-to-bottom; each section is self-contained but builds on prior state.

**Sections:**
1. Setup & imports
2. `bytes_to_unicode` — the byte encoder
3. Pre-tokenization regex
4. `_chunk_key` — text → base symbols
5. `_merge_symbols` — the atomic merge op
6. Stage A in slow-motion (tiny corpus)
7. `_bpe` — heap+linked-list encoder
8. Stage B bigram training
9. Full `encode` → `decode` roundtrip
10. `encode_with_offsets` — NER alignment
11. Load a real trained tokenizer & inspect it
12. Vocabulary statistics & visualisations

---
## 1 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'code'))

import heapq
from collections import Counter
import regex as re
import pickle

# Import the live module so edits to bpe_tokenizer.py are reflected after kernel restart
import importlib
import bpe_tokenizer as _mod
importlib.reload(_mod)

from bpe_tokenizer import (
    BPETokenizer, bytes_to_unicode, get_pairs,
    PRETOKENIZE_PATTERN, PRETOKENIZE_PATTERN_SOCIAL
)

print('Python:', sys.version)
print('OK')

---
## 2 — `bytes_to_unicode`: why every byte gets a printable char

BPE works on **strings**, not bytes.  But raw UTF-8 bytes include control chars (0x00-0x1F) that would silently break string comparison and printing.  
The trick (from GPT-2): **map every byte 0-255 to a printable unicode character** so the byte sequence is a normal string.

> Bytes already in the printable ASCII / Latin-1 ranges keep their natural characters.  
> The remaining 68 bytes get mapped to the block U+0100–U+0143 (never used in normal text).

In [ ]:
b2u = bytes_to_unicode()   # dict: int(byte) -> str(unicode char)

print(f'Total entries: {len(b2u)}  (one per byte value 0-255)')
print()

# Show a few interesting cases
interesting_bytes = [
    (0,    'NUL — control, gets remapped'),
    (32,   'SPACE'),
    (65,   'A — already printable ASCII'),
    (9,    'TAB — control, gets remapped'),
    (195,  'first byte of ü in UTF-8'),
    (188,  'some Latin-1 byte'),
]
for byte_val, note in interesting_bytes:
    char = b2u[byte_val]
    print(f'  byte {byte_val:3d} (0x{byte_val:02X})  ->  repr={repr(char)}  ord={ord(char)}   # {note}')

In [ ]:
# The SPACE byte (32) maps to Ġ (U+0120) — the famous GPT-2 space marker.
# This is what BPETokenizer.space_token is.
space_char = b2u[32]
print(f'space_token = {repr(space_char)}  (U+{ord(space_char):04X})')

# Verify invertibility: decode the whole map back to bytes
u2b = {v: k for k, v in b2u.items()}
assert all(u2b[b2u[i]] == i for i in range(256)), 'not bijective!'
print('Bijective: every byte encodes/decodes cleanly ✓')

In [ ]:
# Encode an arbitrary string to its byte-level token sequence
def show_byte_encode(text, b2u=b2u):
    raw = text.encode('utf-8')
    symbols = [b2u[b] for b in raw]
    print(f'Input:   {repr(text)}')
    print(f'UTF-8:   {list(raw)}')
    print(f'Symbols: {symbols}')
    print(f'(joined as string: {repr("".join(symbols))})')
    return symbols

show_byte_encode('Hello')
print()
show_byte_encode(' Hello')   # leading space -> Ġ prefix
print()
show_byte_encode('naïve')    # multi-byte Unicode

---
## 3 — Pre-tokenization: the regex

Before BPE, text is split into **chunks** (roughly: words, punctuation runs, spaces).  
This is crucial: BPE merges **within** a chunk, never across chunk boundaries — so `"don't"` never accidentally merges the `n` of `don` with the `'` of `'t`.

In [ ]:
pat = re.compile(PRETOKENIZE_PATTERN, re.UNICODE)
pat_social = re.compile(PRETOKENIZE_PATTERN_SOCIAL, re.UNICODE)

def show_pretokenize(text, pattern=pat):
    chunks = pattern.findall(text)
    print(f'Input: {repr(text)}')
    for i, c in enumerate(chunks):
        print(f'  chunk[{i}] = {repr(c)}')
    return chunks

print('=== Standard pattern ===')
show_pretokenize("Hello, world! Don't stop.")
print()
show_pretokenize("naïve café résumé")
print()
show_pretokenize("Price: $1,234.56")

In [ ]:
print('=== Social media pattern (adds @handle and #hashtag rules) ===')
tweet = "@elonmusk #AI is wild lol I can't believe it!!"

print('Standard:')
show_pretokenize(tweet, pat)
print()
print('Social:')
show_pretokenize(tweet, pat_social)

In [ ]:
# Numbers: the pattern groups digits in runs of 1-3
# This prevents "123456789" from becoming one giant token
print('Number grouping (≤3 digits per chunk):')
show_pretokenize("Call 1-800-555-1234 for details.")

---
## 4 — `_chunk_key`: text chunk → base symbol tuple

Each chunk string is converted to a **tuple of byte-encoded symbols** — the starting state for BPE merges.

In [ ]:
tok_scratch = BPETokenizer(vocab_size=500)  # tiny, just for method access

def show_chunk_key(chunk):
    key = tok_scratch._chunk_key(chunk)
    print(f'chunk={repr(chunk)}')
    print(f'  key = {key}')
    # Decode back to show what each symbol represents
    decoded = [bytearray([tok_scratch.byte_decoder[c]]).decode('utf-8', errors='replace') for c in key]
    print(f'  decoded symbols: {decoded}')
    return key

show_chunk_key('Hello')
print()
show_chunk_key(' world')   # leading Ġ from the space byte
print()
show_chunk_key("n't")

---
## 5 — `_merge_symbols`: the atomic merge operation

Given a symbol list and a pair `(a, b)`, replace every adjacent `a b` occurrence with the merged symbol `ab`.

In [ ]:
# Walk through a simple example
symbols = ['H', 'e', 'l', 'l', 'o']
print('Before:', symbols)

step1 = BPETokenizer._merge_symbols(symbols, ('l', 'l'))
print('Merge (l,l):', step1)

step2 = BPETokenizer._merge_symbols(step1, ('H', 'e'))
print('Merge (H,e):', step2)

step3 = BPETokenizer._merge_symbols(step2, ('He', 'll'))
print('Merge (He,ll):', step3)

In [ ]:
# Edge cases
# Pair at end of list
print(BPETokenizer._merge_symbols(['a', 'b', 'c', 'a', 'b'], ('a', 'b')))
# Overlapping: 'a a a' — only non-overlapping left-to-right merges happen
print(BPETokenizer._merge_symbols(['a', 'a', 'a'], ('a', 'a')))

---
## 6 — Stage A in slow-motion: training within-word BPE

Train a tiny tokenizer on a handful of sentences so we can **watch every merge** happen.

In [ ]:
# Monkey-patch _train_within_word to print each merge as it happens

import heapq
from collections import Counter

def train_within_word_verbose(self, line_counts, budget, max_show=30):
    """Identical logic to the real _train_within_word, but prints each merge."""
    word_freqs = Counter()
    line_keys = {}
    for line, freq in line_counts.items():
        keys = [self._chunk_key(chunk) for chunk in self.pat.findall(line)]
        line_keys[line] = keys
        for k in keys:
            word_freqs[k] += freq

    word_syms = [list(k) for k in word_freqs.keys()]
    word_freq = list(word_freqs.values())

    pair_freqs = {}
    pair_to_words = {}
    for idx, syms in enumerate(word_syms):
        f = word_freq[idx]
        for p in zip(syms, syms[1:]):
            pair_freqs[p] = pair_freqs.get(p, 0) + f
            pair_to_words.setdefault(p, set()).add(idx)

    lbpe_exp = getattr(self, '_lbpe_exp', 0.0)
    def _score(p, f):
        return f * (len(p[0]) + len(p[1])) ** lbpe_exp

    heap = [(-_score(p, f), p) for p, f in pair_freqs.items()]
    heapq.heapify(heap)

    def push(p):
        f = pair_freqs.get(p, 0)
        if f > 0:
            heapq.heappush(heap, (-_score(p, f), p))

    rank = 0
    shown = 0
    while len(self.token_to_id) < budget and heap:
        neg_sc, best = heapq.heappop(heap)
        if _score(best, pair_freqs.get(best, 0)) != -neg_sc:
            continue
        if pair_freqs.get(best, 0) < self._min_pair_freq:
            break

        # Decode the pair to readable form
        def sym_readable(s):
            return bytearray(self.byte_decoder[c] for c in s).decode('utf-8', errors='replace')

        freq = pair_freqs[best]
        if shown < max_show:
            merged_surface = sym_readable(best[0] + best[1])
            a_surf = sym_readable(best[0])
            b_surf = sym_readable(best[1])
            print(f'  rank={rank:3d}  freq={freq:4d}  ({repr(a_surf)} + {repr(b_surf)})  ->  {repr(merged_surface)}')
            shown += 1
        elif shown == max_show:
            print(f'  ... (suppressing remaining merges)')
            shown += 1

        self.bpe_ranks[best] = rank
        rank += 1
        self._add_token(best[0] + best[1])

        for idx in list(pair_to_words.get(best, ())):
            syms = word_syms[idx]
            f = word_freq[idx]
            old = Counter(zip(syms, syms[1:]))
            new_syms = self._merge_symbols(syms, best)
            new = Counter(zip(new_syms, new_syms[1:]))
            word_syms[idx] = new_syms
            for p, c in old.items():
                pair_freqs[p] = pair_freqs.get(p, 0) - f * c
                if pair_freqs[p] <= 0:
                    pair_freqs.pop(p, None)
                if p not in new:
                    s = pair_to_words.get(p)
                    if s is not None:
                        s.discard(idx)
                if p != best:
                    push(p)
            for p, c in new.items():
                pair_freqs[p] = pair_freqs.get(p, 0) + f * c
                pair_to_words.setdefault(p, set()).add(idx)
                push(p)

        pair_freqs.pop(best, None)
        pair_to_words.pop(best, None)

    self._atomic = {}
    for idx, key in enumerate(word_freqs):
        if len(word_syms[idx]) == 1:
            self._atomic[key] = word_syms[idx][0]

    print(f'\nStage A done: {rank} merges, vocab size = {len(self.token_to_id)}')
    return line_keys

In [ ]:
SMALL_CORPUS = [
    "the cat sat on the mat",
    "the cat sat on the hat",
    "the cat ate the rat",
    "a fat cat on a flat mat",
    "cats and hats and bats",
    "the cat is the best cat",
] * 10  # repeat 10x so frequencies are meaningful

tiny = BPETokenizer(vocab_size=300)
tiny._auto_configure(SMALL_CORPUS)
tiny._min_pair_freq = 2

# Add base vocab (256 bytes)
for char in tiny.byte_encoder.values():
    tiny._add_token(char)

line_counts = Counter(t for t in SMALL_CORPUS if t.strip())

print('=== Stage A merges (within-word BPE) ===')
line_keys = train_within_word_verbose(tiny, line_counts, budget=290)

In [ ]:
# Inspect the atomic words that Stage A produced (each compressed to 1 token)
print(f'Atomic words (whole word -> single token): {len(tiny._atomic)}')
print()
for key, tok in list(tiny._atomic.items())[:20]:
    surface = bytearray(tiny.byte_decoder[c] for c in tok).decode('utf-8', errors='replace')
    original = bytearray(tiny.byte_decoder[c] for c in key).decode('utf-8', errors='replace')
    print(f'  {repr(original):15s} -> token {repr(tok):20s}  (surface: {repr(surface)})')

---
## 7 — `_bpe`: the heap + doubly-linked-list encoder

At **inference time**, `_bpe` applies the learned merges to a symbol list.  
It uses a **min-heap** keyed by merge rank so the lowest-rank (earliest-learned) pair is always merged first — O(n log n) vs the naive O(n²) GPT-2 approach.

In [ ]:
# Patch _bpe to print each merge step
def _bpe_verbose(self, symbols):
    n = len(symbols)
    if n < 2:
        return list(symbols)

    syms = list(symbols)
    prev = list(range(-1, n - 1))
    nxt  = list(range(1, n + 1))
    alive = [True] * n
    ranks = self.bpe_ranks
    INF = float('inf')

    heap = []
    for i in range(n - 1):
        r = ranks.get((syms[i], syms[i+1]), INF)
        if r < INF:
            heapq.heappush(heap, (r, i))

    step = 0
    while heap:
        r, i = heapq.heappop(heap)
        if not alive[i]: continue
        j = nxt[i]
        if j >= n or not alive[j]: continue
        if ranks.get((syms[i], syms[j]), INF) != r: continue

        a, b = syms[i], syms[j]
        syms[i] = a + b
        alive[j] = False
        nxt[i] = nxt[j]
        if nxt[j] < n:
            prev[nxt[j]] = i

        current = [syms[k] for k in range(n) if alive[k]]

        def decode_sym(s):
            return bytearray(self.byte_decoder[c] for c in s).decode('utf-8', errors='replace')

        print(f'  step {step}: merged rank={r} ({repr(decode_sym(a))} + {repr(decode_sym(b))}) -> {repr(decode_sym(syms[i]))}')
        print(f'           state: {[decode_sym(s) for s in current]}')
        step += 1

        p = prev[i]
        if p >= 0:
            r2 = ranks.get((syms[p], syms[i]), INF)
            if r2 < INF: heapq.heappush(heap, (r2, p))
        k = nxt[i]
        if k < n:
            r2 = ranks.get((syms[i], syms[k]), INF)
            if r2 < INF: heapq.heappush(heap, (r2, i))

    return [syms[i] for i in range(n) if alive[i]]

In [ ]:
# Use the tiny tokenizer trained above to watch a single chunk get BPE'd

chunk = 'cats'
base_symbols = [tiny.byte_encoder[b] for b in chunk.encode('utf-8')]
print(f'Encoding chunk: {repr(chunk)}')
print(f'Base symbols:   {base_symbols}')
print(f'Readable:       {[chr(tiny.byte_decoder[s]) if tiny.byte_decoder[s] < 128 else "?" for s in base_symbols]}')
print()
print('--- BPE merge steps ---')
result = _bpe_verbose(tiny, base_symbols)
print()
print('Final symbols:', result)

In [ ]:
# Try a word NOT in training data — BPE degrades gracefully
chunk = 'zzz'
base_symbols = [tiny.byte_encoder[b] for b in chunk.encode('utf-8')]
print(f'Encoding unseen chunk: {repr(chunk)}')
print(f'Base symbols: {base_symbols}  (no known merges -> stays as individual bytes)')
result = _bpe_verbose(tiny, base_symbols)
print('Final:', result)

---
## 8 — Stage B: bigram training

After Stage A, some whole words are **atomic** (compressed to one token).  
Stage B counts how often adjacent atomic words appear together and creates **cross-word bigram tokens**.

In [ ]:
# Show which atoms are eligible for bigram merges (must contain a letter)
word_atoms = {
    key: tok
    for key, tok in tiny._atomic.items()
    if any(c.isalpha() for c in bytearray(tiny.byte_decoder[c] for c in tok).decode('utf-8', errors='replace'))
}

print(f'Total atomic words: {len(tiny._atomic)}')
print(f'Letter-containing (eligible for bigrams): {len(word_atoms)}')
print()

# Show what pairs actually appear adjacent in the corpus
bigram_freqs = Counter()
for line, freq in line_counts.items():
    atoms = [word_atoms.get(k) for k in line_keys[line]]
    for a, b in zip(atoms, atoms[1:]):
        if a is not None and b is not None:
            bigram_freqs[(a, b)] += freq

print(f'Distinct adjacent atom pairs: {len(bigram_freqs)}')
print()
print('Top bigrams (raw token pair -> surface form -> frequency):')
for (a, b), freq in bigram_freqs.most_common(10):
    def surf(t):
        return bytearray(tiny.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    print(f'  {repr(surf(a)):10s} + {repr(surf(b)):10s}  ->  {repr(surf(a+b)):15s}  freq={freq}')

In [ ]:
# Now run the actual Stage B on the tiny tokenizer
tiny._train_bigrams(line_counts, line_keys)

print(f'Bigram merges learned: {len(tiny.bigram_merges)}')
print()
print('Learned bigrams (surface form, frequency):')
for surface, freq in tiny.get_bigrams()[:15]:
    print(f'  {repr(surface):20s}  freq={freq}')

---
## 9 — Full `encode` → `decode` roundtrip

In [ ]:
# Build a complete small tokenizer the normal way
mini = BPETokenizer(vocab_size=400)
mini.train(SMALL_CORPUS)

print(f'Vocab size: {mini.get_vocab_size()}')
print(f'Stage-A merges: {len(mini.bpe_ranks)}')
print(f'Stage-B bigrams: {len(mini.bigram_merges)}')

In [ ]:
def show_encode(tok, text):
    ids = tok.encode(text)
    tokens = [tok.id_to_token[i] for i in ids]

    def surf(t):
        if t in tok.special_tokens:
            return t
        try:
            return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
        except KeyError:
            return t

    print(f'Input:   {repr(text)}')
    print(f'IDs:     {ids}')
    print(f'Tokens:  {tokens}')
    print(f'Surface: {[surf(t) for t in tokens]}')
    decoded = tok.decode(ids)
    print(f'Decoded: {repr(decoded)}')
    print(f'Lossless: {text == decoded}')
    return ids

print('=== In-vocab sentence ===')
show_encode(mini, 'the cat sat on the mat')
print()
print('=== Out-of-vocab words ===')
show_encode(mini, 'the quantum elephant dances')

In [ ]:
# Demonstrate Stage-B bigram firing during encode
# (Two adjacent atomic words get merged into one bigram token)

print('Learned bigrams:', mini.get_bigrams()[:5])
print()

# Construct a sentence guaranteed to contain the top bigram
if mini.bigram_merges:
    top_pair = list(mini.bigram_merges.keys())[0]
    def surf(t):
        return bytearray(mini.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    word_a = surf(top_pair[0]).strip()
    word_b = surf(top_pair[1]).strip()
    sentence = f'{word_a} {word_b}'
    print(f'Bigram pair surface: "{word_a}" + "{word_b}"  ->  expected single token')
    ids = show_encode(mini, sentence)
    print()
    bigram_tok = mini.bigram_merges[top_pair]
    bigram_id  = mini.token_to_id[bigram_tok]
    print(f'Bigram token id: {bigram_id}  (present in output: {bigram_id in ids})')

---
## 10 — `encode_with_offsets`: NER alignment

NER needs to know which character span each token came from.  
`encode_with_offsets` returns `(token_ids, [(start, end), ...])` where offsets are **character positions** in the original string.

In [ ]:
text = 'the fat cat sat'
ids, offsets = mini.encode_with_offsets(text)

print(f'Input: {repr(text)}')
print()
print(f'{"Token ID":>10}  {"Token surface":>20}  {"Start":>6}  {"End":>4}  Span')
print('-' * 60)
for tid, (start, end) in zip(ids, offsets):
    token = mini.id_to_token.get(tid, '[UNK]')
    try:
        surface = bytearray(mini.byte_decoder[c] for c in token).decode('utf-8', errors='replace')
    except (KeyError, TypeError):
        surface = token
    span_text = text[start:end]
    print(f'{tid:>10}  {repr(surface):>20}  {start:>6}  {end:>4}  {repr(span_text)}')

In [ ]:
# Verify: encode_with_offsets and encode produce the same IDs
ids_a = mini.encode(text)
ids_b, _ = mini.encode_with_offsets(text)
print('IDs match:', ids_a == ids_b)

In [ ]:
# Show how first-subtoken labeling works for NER
# (multi-byte chars, or long words, split into multiple subtokens)

text = 'quantum cats'
ids, offsets = mini.encode_with_offsets(text)

# Simulate NER word-to-token alignment
words = text.split()
word_spans = []
pos = 0
for w in words:
    start = text.index(w, pos)
    word_spans.append((start, start + len(w), w))
    pos = start + len(w)

print('Words:', word_spans)
print()
print('Token -> Word assignment (first-subtoken gets label, rest get -100):')
labels = []
for i, (tid, (ts, te)) in enumerate(zip(ids, offsets)):
    token = mini.id_to_token.get(tid, '?')
    for ws, we, word in word_spans:
        if ts >= ws and te <= we:
            if ts == ws:  # first subtoken of this word
                label = f'B-{word.upper()}'  # fake NER label for demo
                is_first = True
            else:
                label = '-100 (ignored)'
                is_first = False
            print(f'  token[{i}] id={tid:4d} span=({ts},{te}) word={repr(word):10s} label={label}')
            break

---
## 11 — Load a real trained tokenizer & inspect it

In [ ]:
from base_tokenizer import BaseTokenizer

tok1 = BaseTokenizer.load('trained_tokenizers/tokenizer_1.pkl')  # social media
tok2 = BaseTokenizer.load('trained_tokenizers/tokenizer_2.pkl')  # formal/news

for name, tok in [('tokenizer_1 (social)', tok1), ('tokenizer_2 (news)', tok2)]:
    print(f'--- {name} ---')
    print(f'  Vocab size:      {tok.get_vocab_size()}')
    print(f'  Stage-A merges:  {len(tok.bpe_ranks)}')
    print(f'  Stage-B bigrams: {len(tok.bigram_merges)}')
    print(f'  space_token:     {repr(tok.space_token)}')
    print()

In [ ]:
# Top Stage-A merges by rank (rank 0 = first merged = most frequent)
def top_stageA_merges(tok, n=20):
    by_rank = sorted(tok.bpe_ranks.items(), key=lambda kv: kv[1])
    def surf(t):
        return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    return [(rank, surf(a), surf(b), surf(a+b)) for (a,b), rank in by_rank[:n]]

print('=== tokenizer_1 top Stage-A merges ===')
for rank, a, b, merged in top_stageA_merges(tok1):
    print(f'  rank {rank:3d}: {repr(a)} + {repr(b)} -> {repr(merged)}')

print()
print('=== tokenizer_2 top Stage-A merges ===')
for rank, a, b, merged in top_stageA_merges(tok2):
    print(f'  rank {rank:3d}: {repr(a)} + {repr(b)} -> {repr(merged)}')

In [ ]:
# Stage-B bigrams
print('=== tokenizer_1 top bigrams (social) ===')
for surface, freq in tok1.get_bigrams()[:15]:
    print(f'  {repr(surface):30s}  freq={freq}')

print()
print('=== tokenizer_2 top bigrams (news) ===')
for surface, freq in tok2.get_bigrams()[:15]:
    print(f'  {repr(surface):30s}  freq={freq}')

In [ ]:
# Encode real domain sentences
social_tweet = "@elonmusk I can't believe this!!! #AI is wild lol"
news_headline = "Federal Reserve raises interest rates by 25 basis points."

print('=== tokenizer_1 on tweet ===')
show_encode(tok1, social_tweet)
print()

print('=== tokenizer_2 on news ===')
show_encode(tok2, news_headline)

In [ ]:
# Cross-domain: feed a tweet to the news tokenizer
print('=== tokenizer_2 (news) on tweet — domain mismatch ===')
ids = show_encode(tok2, social_tweet)
unk_id = tok2.special_tokens['[UNK]']
unk_count = ids.count(unk_id)
print(f'UNK tokens: {unk_count} / {len(ids)}')
print()
# Byte fallback means 0 UNK — every character is representable via bytes
print('(Byte-level base vocab => OOV rate is always 0, splits to byte tokens instead)')

---
## 12 — Vocabulary statistics & visualisations

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

def token_length_distribution(tok, title):
    """Plot histogram of token surface lengths."""
    lengths = []
    for tid, token in tok.id_to_token.items():
        if token in tok.special_tokens:
            continue
        try:
            surface = bytearray(tok.byte_decoder[c] for c in token).decode('utf-8', errors='replace')
            lengths.append(len(surface))
        except (KeyError, TypeError):
            pass
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(lengths, bins=range(1, max(lengths)+2), edgecolor='black', color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Surface character length')
    ax.set_ylabel('# tokens')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_").replace("/","_")}.png', dpi=100)
    plt.show()
    print(f'  Max length: {max(lengths)}, mean: {sum(lengths)/len(lengths):.1f}')

token_length_distribution(tok1, 'tokenizer_1 social')
token_length_distribution(tok2, 'tokenizer_2 news')

In [ ]:
# Token compression: compare tokens/char on held-out text

def tokens_per_char(tok, text):
    ids = tok.encode(text)
    return len(ids) / len(text)

with open('data/domain_1_dev.txt') as f:
    domain1_dev = f.read()
with open('data/domain_2_dev.txt') as f:
    domain2_dev = f.read()

print('Tokens per character (lower = more compression):')
print(f'  tok1 on domain1 (matched):    {tokens_per_char(tok1, domain1_dev[:5000]):.4f}')
print(f'  tok2 on domain1 (mismatched): {tokens_per_char(tok2, domain1_dev[:5000]):.4f}')
print(f'  tok2 on domain2 (matched):    {tokens_per_char(tok2, domain2_dev[:5000]):.4f}')
print(f'  tok1 on domain2 (mismatched): {tokens_per_char(tok1, domain2_dev[:5000]):.4f}')
print()
print('A matched tokenizer should have FEWER tokens/char (better compression).')

In [ ]:
# Show the first N tokens in the vocabulary (base + first merges)
def show_vocab_slice(tok, start=0, end=280):
    print(f'Tokens [{start}, {end}):')
    for i in range(start, min(end, tok.get_vocab_size())):
        token = tok.id_to_token[i]
        if token in tok.special_tokens:
            surface = token
        else:
            try:
                surface = bytearray(tok.byte_decoder[c] for c in token).decode('utf-8', errors='replace')
            except (KeyError, TypeError):
                surface = token
        if i < 4:
            print(f'  id={i:5d}  {repr(token):30s}  surface={repr(surface)}')
        elif i < 260:
            if i % 32 == 4:
                print(f'  id={i:5d}  {repr(token):30s}  surface={repr(surface)}  (... base byte vocab ...)')
        else:
            print(f'  id={i:5d}  {repr(token):30s}  surface={repr(surface)}')

print('=== tokenizer_2 vocab structure ===')
show_vocab_slice(tok2, start=0, end=275)

In [ ]:
# Merge tree: trace how a given token was built from base bytes
def trace_token(tok, token_str):
    """Show which merges built this token string (Stage A only)."""
    # Invert bpe_ranks: rank -> pair
    by_rank = {v: k for k, v in tok.bpe_ranks.items()}

    def surf(t):
        try:
            return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
        except (KeyError, TypeError):
            return t

    def _trace(sym, depth=0):
        indent = '  ' * depth
        print(f'{indent}{repr(surf(sym))}')
        # Find the merge that produced this symbol
        for rank in range(len(by_rank)):
            a, b = by_rank[rank]
            if a + b == sym and len(sym) > 1:
                print(f'{indent}= ({repr(surf(a))} + {repr(surf(b))})  [rank {rank}]')
                _trace(a, depth + 1)
                _trace(b, depth + 1)
                return
        # Base case: single byte
        print(f'{indent}[base byte: {repr(surf(sym))}]')

    print(f'Merge tree for token {repr(surf(token_str))}:')
    _trace(token_str)

def surf2(t, tok=tok2):
    try:
        return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    except Exception:
        return t

# Pick a few interesting Stage-A tokens (medium length, all-alpha surface form)
interesting = [
    t for t in tok2.id_to_token.values()
    if t not in tok2.special_tokens
    and 4 <= len(t) <= 8
    and surf2(t).strip().isalpha()
][:3]

for tok_str in interesting:
    trace_token(tok2, tok_str)
    print()

In [ ]:
# Cache efficiency demo: second encode is O(1) from cache
import time

text = domain2_dev[:2000]

# Clear cache
tok2._encode_cache.clear()
tok2.cache.clear()

t0 = time.perf_counter()
_ = tok2.encode(text)
t1 = time.perf_counter()
_ = tok2.encode(text)
t2 = time.perf_counter()

print(f'First encode  (cold cache): {(t1-t0)*1000:.2f} ms')
print(f'Second encode (warm cache): {(t2-t1)*1000:.2f} ms')
print(f'Speedup: {(t1-t0)/(t2-t1):.0f}x')

In [ ]:
# _auto_configure: what it decides for each domain
with open('data/domain_1_train.txt') as f:
    d1 = f.readlines()
with open('data/domain_2_train.txt') as f:
    d2 = f.readlines()

for name, texts in [('domain_1 (social)', d1[:500]), ('domain_2 (news)', d2[:500])]:
    step = max(1, len(texts) // 2000)
    sample = [t for t in texts[::step] if t.strip()][:2000]
    total_chars = sum(len(t) for t in sample)
    at_hash = sum(t.count('@') + t.count('#') for t in sample)
    short_frac = sum(1 for t in sample if len(t.rstrip()) < 80) / len(sample)
    at_hash_density = at_hash / total_chars if total_chars else 0
    is_social = at_hash_density > 0.005 or short_frac > 0.65
    print(f'{name}:')
    print(f'  @/# density: {at_hash_density:.4f}  short_frac: {short_frac:.3f}  is_social={is_social}')